# WordPiece

**Audience:** complete beginners who finished the BPE notebook.

WordPiece looks like BPE on the surface: start from characters, glue pairs, grow a
vocabulary. The **score** is different. BPE picks the pair that occurs most often.
WordPiece picks the pair that is most “surprising” given the parts — a likelihood ratio.

Google built WordPiece for speech, then BERT made it famous. Continuation pieces inside a
word are marked with `##` so the decoder knows they do not start a new word.


## Learning path

```mermaid
flowchart LR
  bpe[BPE: max frequency] --> wp[WordPiece: max score]
  wp --> marks["## continuation marks"]
  marks --> greedy[Greedy longest-match encode]
  greedy --> hf[HuggingFace WordPieceTrainer]
```


## BPE vs WordPiece in one picture

```mermaid
flowchart TD
  subgraph bpe [BPE]
    b1[Count pair AB] --> b2["Pick max count(AB)"]
  end
  subgraph wp [WordPiece]
    w1["count(AB) / count(A) / count(B)"] --> w2[Pick max score]
  end
  bpe --> same[Both glue AB into a new vocab piece]
  wp --> same
```

If `t` and `h` are already extremely common, BPE still loves merging `th` because it appears
a lot. WordPiece asks: *does `th` occur more than you'd expect from `t` and `h` alone?*
That is why BERT pieces often look a bit more “word-like”.


## Setup


In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets

def find_root() -> Path:
    here = Path.cwd()
    for candidate in [here, here.parent]:
        if (candidate / "data" / "tiny_corpus.txt").exists():
            return candidate
    raise FileNotFoundError("Run the notebook from the repo root or the notebooks/ folder.")

ROOT = find_root()
CORPUS = ROOT / "data" / "tiny_corpus.txt"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print(f"corpus: {CORPUS}")


corpus: /Users/sourangshupal/Downloads/tokenization-explainer/data/tiny_corpus.txt


## Toy WordPiece trainer

We reuse the BPE bookkeeping (splits + pair counts) and only change the **ranking**.

Score of pair `(a, b)`:

\[
\mathrm{score}(a,b) = \frac{\mathrm{freq}(a,b)}{\mathrm{freq}(a)\,\mathrm{freq}(b)}
\]

High score: `a` and `b` stick together more than chance. Low score: they just happen to
sit next to each other because both are common.

If you dump the whole grab-bag file into this scorer, the first merges become `qu`, `ju`,
and then the unique word `vocabulary` — rare letters make the denominator tiny. That is
honest WordPiece, and it is why production models train on Wikipedia-scale text. We train
the hand-rolled model on the classic four-word toy set from the BPE paper so you can
actually watch `low` / `lower` / `newest` / `widest`. HuggingFace on the full file comes
later.


In [2]:
def load_word_counts(path: Path) -> Counter[str]:
    counts: Counter[str] = Counter()
    for line in path.read_text(encoding="utf-8").splitlines():
        for raw in line.lower().split():
            word = "".join(ch for ch in raw if ch.isalpha())
            if word:
                counts[word] += 1
    return counts


def initial_splits(counts: Counter[str]) -> dict[str, list[str]]:
    return {word: list(word) for word in counts}


def symbol_freqs(splits: dict[str, list[str]], counts: Counter[str]) -> Counter[str]:
    freq: Counter[str] = Counter()
    for word, n in counts.items():
        for sym in splits[word]:
            freq[sym] += n
    return freq


def pair_counts(splits: dict[str, list[str]], counts: Counter[str]) -> Counter[tuple[str, str]]:
    pairs: Counter[tuple[str, str]] = Counter()
    for word, n in counts.items():
        symbols = splits[word]
        for left, right in zip(symbols, symbols[1:]):
            pairs[(left, right)] += n
    return pairs


def apply_merge(splits: dict[str, list[str]], pair: tuple[str, str]) -> dict[str, list[str]]:
    a, b = pair
    glued = a + b
    updated: dict[str, list[str]] = {}
    for word, symbols in splits.items():
        out: list[str] = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                out.append(glued)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        updated[word] = out
    return updated


word_counts = Counter({"low": 5, "lower": 2, "newest": 3, "widest": 2})
print("classic toy counts:", dict(word_counts))
print("full-file types (used later by HuggingFace):", len(load_word_counts(CORPUS)))


classic toy counts: {'low': 5, 'lower': 2, 'newest': 3, 'widest': 2}
full-file types (used later by HuggingFace): 105


In [3]:
def best_wordpiece_pair(
    splits: dict[str, list[str]],
    counts: Counter[str],
    min_freq: int = 2,
) -> tuple[tuple[str, str], float, int] | None:
    # Rank pairs by likelihood score, ignoring hapax pairs.
    # On a tiny corpus, raw argmax(score) prefers rare letters like q+u
    # because freq(q)*freq(u) is tiny. Real WordPiece uses a count floor
    # (and a huge corpus). We keep the floor so later merges still happen.
    pairs = pair_counts(splits, counts)
    singles = symbol_freqs(splits, counts)
    ranked: list[tuple[tuple[str, str], float, int]] = []
    for pair, freq in pairs.items():
        if freq < min_freq:
            continue
        a, b = pair
        score = freq / (singles[a] * singles[b])
        ranked.append((pair, score, freq))
    if not ranked:
        return None
    ranked.sort(key=lambda row: row[1], reverse=True)
    return ranked[0]


def train_wordpiece(counts: Counter[str], num_merges: int) -> list[str]:
    splits = initial_splits(counts)
    vocab: set[str] = set()
    for symbols in splits.values():
        vocab.update(symbols)

    for step in range(1, num_merges + 1):
        ranked = best_wordpiece_pair(splits, counts)
        if ranked is None:
            print(f"stop at step {step}: no pair left with frequency >= 2")
            break
        pair, score, freq = ranked
        a, b = pair
        glued = a + b
        vocab.add(glued)
        splits = apply_merge(splits, pair)
        print(f"{step:02d}. {a!r} + {b!r} → {glued!r}   score={score:.4f}  pair_freq={freq}")
    return sorted(vocab, key=lambda s: (len(s), s))


wp_vocab = train_wordpiece(word_counts, num_merges=25)
print(f"\nvocab size {len(wp_vocab)}")
print("longest pieces:", sorted(wp_vocab, key=len, reverse=True)[:12])


01. 'i' + 'd' → 'id'   score=0.5000  pair_freq=2
02. 's' + 't' → 'st'   score=0.2000  pair_freq=5
03. 'l' + 'o' → 'lo'   score=0.1429  pair_freq=7
04. 'e' + 'r' → 'er'   score=0.1000  pair_freq=2
05. 'n' + 'e' → 'ne'   score=0.1250  pair_freq=3
06. 'e' + 'st' → 'est'   score=0.2000  pair_freq=5
07. 'id' + 'est' → 'idest'   score=0.2000  pair_freq=2
08. 'lo' + 'w' → 'low'   score=0.0833  pair_freq=7
09. 'ne' + 'w' → 'new'   score=0.2000  pair_freq=3
10. 'w' + 'idest' → 'widest'   score=0.5000  pair_freq=2
11. 'new' + 'est' → 'newest'   score=0.3333  pair_freq=3
12. 'low' + 'er' → 'lower'   score=0.1429  pair_freq=2
stop at step 13: no pair left with frequency >= 2

vocab size 22
longest pieces: ['newest', 'widest', 'idest', 'lower', 'est', 'low', 'new', 'er', 'id', 'lo', 'ne', 'st']


## Encoding with `##`

BERT-style WordPiece does **not** replay merges. It uses **greedy longest-match** on each
word:

1. Look at the whole word. If it is in the vocab, emit it.
2. If not, find the longest prefix that is in the vocab.
3. The remainder must be found with a `##` prefix (`est` → `##est`).
4. If a leftover character is missing, emit `[UNK]` for the whole word (BERT's original
   behaviour) or skip — we mark `[UNK]` so you can see the failure.

```mermaid
flowchart TD
  word[Input word] --> whole{Whole word in vocab?}
  whole -->|yes| emit[Emit word]
  whole -->|no| prefix[Longest prefix in vocab]
  prefix --> rest[Remainder]
  rest --> hash["Look up ##remainder"]
  hash --> more{More leftover?}
  more -->|yes| prefix
  more -->|no| done[Done]
```


In [4]:
def wordpiece_tokenize_word(word: str, vocab: set[str]) -> list[str]:
    # Greedy longest-match. Non-initial pieces are printed with ##.
    word = word.lower()
    pieces: list[str] = []
    start = 0
    while start < len(word):
        end = len(word)
        found: str | None = None
        while end > start:
            piece = word[start:end]
            if piece in vocab:
                found = piece
                break
            end -= 1
        if found is None:
            return ["[UNK]"]
        pieces.append(found if start == 0 else f"##{found}")
        start += len(found)
    return pieces


LOOKUP = set(wp_vocab)

for sample in ["low", "lower", "newest", "tokenization", "unhappiness", "xyzzy"]:
    print(f"{sample:15} → {wordpiece_tokenize_word(sample, LOOKUP)}")


low             → ['low']
lower           → ['lower']
newest          → ['newest']
tokenization    → ['[UNK]']
unhappiness     → ['[UNK]']
xyzzy           → ['[UNK]']


## Interactive playground


In [5]:
box = widgets.Text(
    value="lower newest unhappiness",
    description="Text:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "50px"},
)
out = widgets.Output()


def encode_sentence(text: str) -> list[str]:
    pieces: list[str] = []
    for raw in text.split():
        word = "".join(ch for ch in raw.lower() if ch.isalpha())
        if word:
            pieces.extend(wordpiece_tokenize_word(word, LOOKUP))
    return pieces


def _run(_change=None) -> None:
    with out:
        out.clear_output()
        pieces = encode_sentence(box.value)
        print("pieces:", pieces)
        print("count: ", len(pieces))


box.observe(_run, names="value")
_run()
display(box, out)


Text(value='lower newest unhappiness', description='Text:', layout=Layout(width='90%'), style=TextStyle(descri…

Output()

## Side-by-side with BPE on one word

Same toy corpus, two scoring rules. They often agree on this tiny file. Differences show up
on real Wikipedia-scale data. The `##` mark is the part you will always notice in BERT.


In [6]:
def bpe_style_splits(counts: Counter[str], num_merges: int) -> dict[str, list[str]]:
    splits = {w: list(w) + ["</w>"] for w in counts}

    def pairs(sp):
        c: Counter[tuple[str, str]] = Counter()
        for word, n in counts.items():
            s = sp[word]
            for a, b in zip(s, s[1:]):
                c[(a, b)] += n
        return c

    def merge(sp, pair):
        a, b = pair
        glued = a + b
        out = {}
        for word, symbols in sp.items():
            row, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                    row.append(glued)
                    i += 2
                else:
                    row.append(symbols[i])
                    i += 1
            out[word] = row
        return out

    for _ in range(num_merges):
        stats = pairs(splits)
        if not stats:
            break
        splits = merge(splits, stats.most_common(1)[0][0])
    return splits


bpe_splits = bpe_style_splits(word_counts, 20)
word = "lower"
print("WordPiece greedy:", wordpiece_tokenize_word(word, LOOKUP))
print("BPE after 20 merges (with </w>):", bpe_splits.get(word, "not in corpus as a type"))
print("WordPiece on 'lowest':", wordpiece_tokenize_word("lowest", LOOKUP))
print("BPE on corpus type 'lowest':", bpe_splits.get("lowest", "lowest was not a training type"))


WordPiece greedy: ['lower']
BPE after 20 merges (with </w>): ['lower</w>']
WordPiece on 'lowest': ['low', '##est']
BPE on corpus type 'lowest': lowest was not a training type


## Production library: HuggingFace `WordPieceTrainer`

`tokenizers` 0.23 trains WordPiece and adds `##` for you. Pre-tokenization is still
whitespace — BERT also uses a punctuation splitter in the full pipeline. We keep whitespace
so the lesson stays aligned with BPE.


In [7]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer

hf_wp = Tokenizer(WordPiece(unk_token="[UNK]"))
hf_wp.pre_tokenizer = Whitespace()
trainer = WordPieceTrainer(
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    vocab_size=120,
    min_frequency=1,
    show_progress=False,
)
hf_wp.train([str(CORPUS)], trainer)

for sample in ["lower newest", "unhappiness", "tokenization", "xyzzy"]:
    enc = hf_wp.encode(sample)
    print(f"{sample:20} {enc.tokens}")

hf_wp.save(str(ARTIFACTS / "hf_wordpiece.json"))
print("vocab size", hf_wp.get_vocab_size())


lower newest         ['lower', 'newest']
unhappiness          ['un', '##h', '##ap', '##p', '##in', '##es', '##s']
tokenization         ['tokeniz', '##at', '##i', '##o', '##n']
xyzzy                ['x', '##y', '##z', '##z', '##y']
vocab size 120


## Where you will see WordPiece

- **BERT**, DistilBERT, Electra — `##` pieces, `[CLS]` / `[SEP]` / `[MASK]`
- **Original WordPiece** at Google — speech + multilingual search

If you see `##ing` or `##ization` in a model card, you are looking at WordPiece (or a
BERT-compatible clone), not GPT BPE.


## Exercises

1. In the from-scratch trainer, print the **top 5 BPE pairs** and the **top 5 WordPiece
   scores** at step 1 (before any merge). Do they pick the same winner?
2. Why does BERT use `##` instead of `</w>`? Which one makes detokenization easier?
3. Encode `playing` and `played` with the HuggingFace WordPiece model. Did they share a stem?
4. What should happen if you encode a character the vocab never saw? Check with `xyzzy`.
